# **Working with JSON Data**

![](../images/JSON_object.png)

In [1]:
import json

In [2]:
sample_json = """{
    "id": 143,
    "name": "Silicon Valley",
    "type": "Scripted",
    "language": "English",
    "genres": [
        "Comedy"
    ],
    "network": {
        "id": 8,
        "name": "HBO",
        "country": {
        "name": "United States",
        "code": "US",
        "timezone": "America/New_York"
        }
    }
}"""

In [3]:
document = json.loads(sample_json)

document

{'id': 143,
 'name': 'Silicon Valley',
 'type': 'Scripted',
 'language': 'English',
 'genres': ['Comedy'],
 'network': {'id': 8,
  'name': 'HBO',
  'country': {'name': 'United States',
   'code': 'US',
   'timezone': 'America/New_York'}}}

In [4]:
type(document)

dict

In [5]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [6]:
spark = SparkSession.builder.getOrCreate()

spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/13 11:14:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=pyspark-shell>

In [7]:
document = spark.read.json('../data/shows/shows-silicon-valley.json')

document.count()

1

In [8]:
documents = spark.read.json(
    '../data/shows/shows-*.json', 
    multiLine=True)

documents.count()

26/11/13 11:14:49 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/shows/shows-*.json.
java.io.FileNotFoundException: File ../data/shows/shows-*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.an

3

In [9]:
documents.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

In [10]:
documents.columns

['_embedded',
 '_links',
 'externals',
 'genres',
 'id',
 'image',
 'language',
 'name',
 'network',
 'officialSite',
 'premiered',
 'rating',
 'runtime',
 'schedule',
 'status',
 'summary',
 'type',
 'updated',
 'url',
 'webChannel',
 'weight']

## **The Array Type**

In [11]:
array_subset = document.select('name', 'genres')

array_subset.show(1, False)

+--------------+--------+
|name          |genres  |
+--------------+--------+
|Silicon Valley|[Comedy]|
+--------------+--------+



![](../images/array_subset.png)

In [12]:
array_subset = array_subset.select(
    'name',
    array_subset.genres[0].alias('dot_and_index'),
    F.col('genres')[0].alias('col_and_index'),
    array_subset.genres.getItem(0).alias('dot_and_method'),
    F.col('genres').getItem(0).alias('col_and_method')
)

array_subset.show(truncate=False)

+--------------+-------------+-------------+--------------+--------------+
|name          |dot_and_index|col_and_index|dot_and_method|col_and_method|
+--------------+-------------+-------------+--------------+--------------+
|Silicon Valley|Comedy       |Comedy       |Comedy        |Comedy        |
+--------------+-------------+-------------+--------------+--------------+



In [13]:
array_subset_repeated = array_subset.select(
    'name',
    F.lit('Comedy').alias('one'),
    F.lit('Horror').alias('two'),
    F.lit('Drama').alias('three'),
    F.col('dot_and_index')
)

array_subset_repeated.show()

+--------------+------+------+-----+-------------+
|          name|   one|   two|three|dot_and_index|
+--------------+------+------+-----+-------------+
|Silicon Valley|Comedy|Horror|Drama|       Comedy|
+--------------+------+------+-----+-------------+



In [14]:
array_subset_repeated = array_subset_repeated.select(
    'name',
    F.array('one', 'two', 'three').alias('Some_Genres'),
    F.array_repeat('dot_and_index', 5).alias('Repeated_Genres')
)

array_subset_repeated.show(truncate=False)

+--------------+-----------------------+----------------------------------------+
|name          |Some_Genres            |Repeated_Genres                         |
+--------------+-----------------------+----------------------------------------+
|Silicon Valley|[Comedy, Horror, Drama]|[Comedy, Comedy, Comedy, Comedy, Comedy]|
+--------------+-----------------------+----------------------------------------+



In [15]:
array_subset_repeated.select(
    'name',
    F.size('Some_Genres'),
    F.size('Repeated_Genres')
).show()

+--------------+-----------------+---------------------+
|          name|size(Some_Genres)|size(Repeated_Genres)|
+--------------+-----------------+---------------------+
|Silicon Valley|                3|                    5|
+--------------+-----------------+---------------------+



In [16]:
array_subset_repeated.select(
    'name',
    F.array_distinct('Some_Genres'),
    F.array_distinct('Repeated_Genres')
).show(1, truncate=False)

+--------------+---------------------------+-------------------------------+
|name          |array_distinct(Some_Genres)|array_distinct(Repeated_Genres)|
+--------------+---------------------------+-------------------------------+
|Silicon Valley|[Comedy, Horror, Drama]    |[Comedy]                       |
+--------------+---------------------------+-------------------------------+



In [17]:
array_subset_repeated = array_subset_repeated.select(
    'name',
    F.array_intersect('Some_Genres', 'Repeated_Genres').alias('Genres')
)

array_subset_repeated.show()

+--------------+--------+
|          name|  Genres|
+--------------+--------+
|Silicon Valley|[Comedy]|
+--------------+--------+



In [18]:
array_subset_repeated.select(
    'Genres',
    F.array_position('Genres', 'Comedy')
).show()

+--------+------------------------------+
|  Genres|array_position(Genres, Comedy)|
+--------+------------------------------+
|[Comedy]|                             1|
+--------+------------------------------+



## **The Map Type**

In [19]:
columns = ["name", "language", "type"]

df_map = documents.select(
    *[F.lit(col) for col in columns],
    F.array(*columns).alias('values')
)

df_map.show(3, False)

+----+--------+----+-------------------------------------+
|name|language|type|values                               |
+----+--------+----+-------------------------------------+
|name|language|type|[The Golden Girls, English, Scripted]|
|name|language|type|[Breaking Bad, English, Scripted]    |
|name|language|type|[Silicon Valley, English, Scripted]  |
+----+--------+----+-------------------------------------+



In [20]:
df_map = df_map.select(
    F.array(*columns).alias('keys'), 'values'
)

df_map.show(3, False)

+----------------------+-------------------------------------+
|keys                  |values                               |
+----------------------+-------------------------------------+
|[name, language, type]|[The Golden Girls, English, Scripted]|
|[name, language, type]|[Breaking Bad, English, Scripted]    |
|[name, language, type]|[Silicon Valley, English, Scripted]  |
+----------------------+-------------------------------------+



In [21]:
df_map = df_map.select(
    F.map_from_arrays('keys', 'values').alias('mapped')
)

df_map.printSchema()

root
 |-- mapped: map (nullable = false)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [22]:
df_map.show(3, False)

+-----------------------------------------------------------------+
|mapped                                                           |
+-----------------------------------------------------------------+
|{name -> The Golden Girls, language -> English, type -> Scripted}|
|{name -> Breaking Bad, language -> English, type -> Scripted}    |
|{name -> Silicon Valley, language -> English, type -> Scripted}  |
+-----------------------------------------------------------------+



In [23]:
df_map.select(
    F.col('mapped.name'),
    F.col('mapped')['name'],
    df_map.mapped['name']
).show()

+----------------+----------------+----------------+
|            name|    mapped[name]|    mapped[name]|
+----------------+----------------+----------------+
|The Golden Girls|The Golden Girls|The Golden Girls|
|    Breaking Bad|    Breaking Bad|    Breaking Bad|
|  Silicon Valley|  Silicon Valley|  Silicon Valley|
+----------------+----------------+----------------+



## **The Struct Type**

In [24]:
documents.select('schedule').printSchema()

root
 |-- schedule: struct (nullable = true)
 |    |-- days: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- time: string (nullable = true)



![](../images/struct_vis.png)

In [25]:
documents.select(F.col("_embedded")).printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

![](../images/_embedded_field.png)

In [26]:
documents_clean = documents.withColumn(
    'episodes', F.col('_embedded.episodes')
).drop('_embedded')

documents_clean.printSchema()

root
 |-- _links: struct (nullable = true)
 |    |-- previousepisode: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |    |-- self: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |-- externals: struct (nullable = true)
 |    |-- imdb: string (nullable = true)
 |    |-- thetvdb: long (nullable = true)
 |    |-- tvrage: long (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- id: long (nullable = true)
 |-- image: struct (nullable = true)
 |    |-- medium: string (nullable = true)
 |    |-- original: string (nullable = true)
 |-- language: string (nullable = true)
 |-- name: string (nullable = true)
 |-- network: struct (nullable = true)
 |    |-- country: struct (nullable = true)
 |    |    |-- code: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- timezone: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nul

In [27]:
episodes_name = documents_clean.select(F.col('episodes.name'))

episodes_name.printSchema()

root
 |-- name: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [28]:
episodes_name.select(
    F.explode('name').alias('name')
).show(3, False)

+----------------------------------+
|name                              |
+----------------------------------+
|The Engagement                    |
|Guess Who's Coming to the Wedding?|
|Rose the Prude                    |
+----------------------------------+
only showing top 3 rows


In Apache Spark, **Complex Data Types** (`array`, `map`, and `struct`) allow you to handle semi-structured, nested, or multi-valued data directly inside tabular DataFrames without needing to flatten everything into rigid, flat flat-file structures.

Under Project Tungsten and the Catalyst Optimizer, these complex types are stored compactly in binary off-heap memory formats, minimizing Java Garbage Collection (GC) overhead while providing rich SQL-like manipulation functions.

---

## **1. Array (`ArrayType`)**

An **Array** is an ordered collection of elements where **all elements must share the exact same data type** (e.g., an array of strings, an array of integers).

### **⚙️ Characteristics**

* **Homogeneous:** Every item in the array must be of the same type.
* **Ordered:** Elements maintain their insertion or specified sequence, accessible via 1-based indexing (e.g., `element_at(col, 1)`) or 0-based indexing depending on the specific function.
* **Nullability:** Arrays can contain `null` elements, and the array itself can be null.
* **Duplicates:** Duplicate values are allowed.

### **🛠️ Common Usage & PySpark Example**

Arrays are ideal for storing multi-valued attributes like tags, lists of product IDs, clickstream path histories, or tokenized words.

```python
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ComplexTypesExample").getOrCreate()

# Sample data with an array of tags and scores
data = [
    (1, "Alice", ["python", "spark", "sql"]),
    (2, "Bob", ["java", "spring"]),
]
df = spark.createDataFrame(data, ["id", "name", "skills"])

# 1. Check if array contains a specific value
df_filtered = df.filter(F.array_contains(F.col("skills"), "spark"))

# 2. Explode the array into individual rows (one row per skill)
df_exploded = df.select("id", "name", F.explode("skills").alias("skill"))
df_exploded.show(truncate=False)

```

---

## **2. Map (`MapType`)**

A **Map** is a collection of key-value pairs (dictionary structure). **All keys must share the same data type**, and **all values must share the same data type**.

### **⚙️ Characteristics**

* **Key-Value Pair Structure:** Data is looked up by key rather than index position.
* **Homogeneous Types:** All keys share a common type (frequently `StringType`), and all values share a common type.
* **Key Constraints:** Keys **cannot be null**. Values can be null.
* **Unordered:** Maps do not guarantee an ordered sequence of elements.

### **🛠️ Common Usage & PySpark Example**

Maps are powerful for dynamic attributes, flexible telemetry metadata, configuration settings, or multi-language translation dictionaries where keys vary per row.

```python
# Sample data with a map of metadata attributes
data = [
    (1, {"env": "prod", "version": "v1.2", "region": "us-east"}),
    (2, {"env": "dev", "version": "v2.0"}),
]
df = spark.createDataFrame(data, ["id", "metadata"])

# 1. Access a specific map value by key using bracket notation or element_at
df_env = df.select("id", df.metadata["env"].alias("environment"))

# 2. Extract all keys or values as arrays
df_keys = df.select("id", F.map_keys("metadata").alias("all_keys"))

df_env.show(truncate=False)

```

---

## **3. Struct (`StructType`)**

A **Struct** is a record type containing a sequence of named fields with associated data types. It is structurally identical to a row in a table, a JSON object, or a nested Python dictionary.

### **⚙️ Characteristics**

* **Heterogeneous Fields:** Unlike arrays and maps, fields within a struct **can have completely different data types** (e.g., `Field1` is an integer, `Field2` is a string, `Field3` is an array).
* **Named Access:** Fields are accessed by name (dot notation like `struct_col.field_name`) or position.
* **Building Block of Nesting:** Structs form the backbone of nested DataFrames when reading JSON, Parquet, or Delta Lake formats with hierarchical schemas.

### **🛠️ Common Usage & PySpark Example**

Structs are used to group related attributes together—such as bundling `street`, `city`, and `zipcode` into an `address` struct column.

```python
# Sample data with a nested struct for address
data = [
    (1, "Alice", {"city": "Paris", "zip": "75001"}),
    (2, "Bob", {"city": "Berlin", "zip": "10115"}),
]
# Schema is automatically inferred as StructType containing 'city' and 'zip'
df = spark.createDataFrame(data, ["id", "name", "address"])

# 1. Access nested fields using dot notation
df_city = df.select("name", F.col("address.city").alias("city"))

# 2. Flatten/expand all fields of a struct using the star selector (*)
df_unpacked = df.select("id", "name", "address.*")

df_unpacked.show(truncate=False)

```

---

### **📊 Summary Comparison Matrix**

| Feature | **Array (`ArrayType`)** | **Map (`MapType`)** | **Struct (`StructType`)** |
| --- | --- | --- | --- |
| **Data Structure** | Ordered list of items | Key-value pairs dictionary | Collection of named fields |
| **Data Types** | **Homogeneous** (All items same type) | **Homogeneous keys**, **Homogeneous values** | **Heterogeneous** (Fields can have unique types) |
| **Null Keys Allowed?** | N/A (Index-based) | **No** (Keys cannot be null) | N/A (Field names are static strings) |
| **Access Pattern** | By index position or iteration | By key lookup (`map['key']`) | By dot notation (`struct.field`) |
| **Primary Use Case** | Tags, lists, history logs | Flexible attributes, tags, config | Nested entities, JSON records, rows |

---

### **⚠️ Performance & Architecture Considerations**

1. **Explosion Overhead (`explode`):** Using functions like `F.explode()` on arrays or maps performs a lateral join under the hood, duplicating parent row data for every element in the collection. On massive datasets, this can cause a massive data expansion and trigger expensive network shuffles.
2. **Pushdown Limitations:** Traditional storage layers struggle to push down predicate filters inside deeply nested maps or arrays compared to top-level flat columns. When designing performance-critical tables, frequently queried attributes should be kept as flat top-level columns rather than buried inside maps or arrays.

In [30]:
import pyspark.sql.types as T

In [50]:
episode_links_schema = T.StructType(
    [
        T.StructField(
            'self', T.StructType([T.StructField('href', T.StringType())])
        )
    ]
)

episode_image_schema = T.StructType(
    [
        T.StructField('meduim', T.StringType()),
        T.StructField('original', T.StringType())
    ]
)

episodes_schema = T.StructType(
    [
        T.StructField('_links', episode_links_schema),
        T.StructField('airdate', T.DateType()),
        T.StructField('airstamp', T.TimestampType()),
        T.StructField("airtime", T.StringType()),
        T.StructField("id", T.StringType()),
        T.StructField("image", episode_image_schema),
        T.StructField("name", T.StringType()),
        T.StructField("number", T.LongType()),
        T.StructField("runtime", T.LongType()),
        T.StructField("season", T.LongType()),
        T.StructField("summary", T.StringType()),
        T.StructField("url", T.StringType()),
    ]
)

embedded_schema = T.StructType(
    [
        T.StructField(
            '_embedded', T.StructType(
                [
                    T.StructField(
                       'episodes', T.ArrayType(episodes_schema) 
                    )
                ]
            )
        )
    ]
)

In [33]:
documents.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

In [51]:
document_with_schema = spark.read.json(
    '../data/shows/shows-silicon-valley.json',
    schema=embedded_schema,
    mode='FAILFAST'
)

In [52]:
document_with_schema.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: date (nullable = true)
 |    |    |    |-- airstamp: timestamp (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- meduim: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = t

In [53]:
for col in ['airdate', 'airstamp']:
    document_with_schema.select(
        f"_embedded.episodes.{col}"
    ).select(
        F.explode(col)
    ).show(5)

+----------+
|       col|
+----------+
|2014-04-06|
|2014-04-13|
|2014-04-20|
|2014-04-27|
|2014-05-04|
+----------+
only showing top 5 rows
+-------------------+
|                col|
+-------------------+
|2014-04-07 02:00:00|
|2014-04-14 02:00:00|
|2014-04-21 02:00:00|
|2014-04-28 02:00:00|
|2014-05-05 02:00:00|
+-------------------+
only showing top 5 rows


In [46]:
from py4j.protocol import Py4JJavaError

In [49]:
episodes_schema_BAD = T.StructType(
    [
        T.StructField('_links', episode_links_schema),
        T.StructField('airdate', T.DateType()),
        T.StructField('airstamp', T.TimestampType()),
        T.StructField("airtime", T.StringType()),
        T.StructField("id", T.StringType()),
        T.StructField("image", episode_image_schema),
        T.StructField("name", T.StringType()),
        T.StructField("number", T.LongType()),
        T.StructField("runtime", T.LongType()),
        T.StructField("season", T.LongType()),
        T.StructField("summary", T.LongType()),
        T.StructField("url", T.LongType()),
    ]
)

embedded_schema2 = T.StructType(
    [
        T.StructField(
            '_embedded', T.StructType(
                [
                    T.StructField(
                       'episodes', T.ArrayType(episodes_schema_BAD) 
                    )
                ]
            )
        )
    ]
)

document_with_schema_wrong = spark.read.json(
    "../data/shows/shows-silicon-valley.json",
    schema=embedded_schema2,
    mode="FAILFAST",
)

try:
    document_with_schema_wrong.show()
except Py4JJavaError:
    pass

26/11/13 12:09:36 ERROR Executor: Exception in task 0.0 in stage 35.0 (TID 39)
org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:///workspace/data/shows/shows-silicon-valley.json.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:940)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:149)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:44)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory

In [54]:
import pprint

In [56]:
pprint.pprint(
    document_with_schema.select(
        F.explode("_embedded.episodes").alias('episode')
    )
    .select('episode.airtime')
    .schema.jsonValue()
)

{'fields': [{'metadata': {},
             'name': 'airtime',
             'nullable': True,
             'type': 'string'}],
 'type': 'struct'}


In [57]:
pprint.pprint(
    T.StructField(
        "array_example", 
        T.ArrayType(T.StringType())
    ).jsonValue()
)

{'metadata': {},
 'name': 'array_example',
 'nullable': True,
 'type': {'containsNull': True, 'elementType': 'string', 'type': 'array'}}


In [58]:
pprint.pprint(
    T.StructField(
        "map_example", 
        T.MapType(T.StringType(), T.LongType())
    ).jsonValue()
)

{'metadata': {},
 'name': 'map_example',
 'nullable': True,
 'type': {'keyType': 'string',
          'type': 'map',
          'valueContainsNull': True,
          'valueType': 'long'}}


In [59]:
pprint.pprint(
    T.StructType(
        [
            T.StructField(
                "map_example", 
                T.MapType(T.StringType(), T.LongType())
            ),
            T.StructField(
                "array_example", 
                T.ArrayType(T.StringType())),
        ]).jsonValue()
)

{'fields': [{'metadata': {},
             'name': 'map_example',
             'nullable': True,
             'type': {'keyType': 'string',
                      'type': 'map',
                      'valueContainsNull': True,
                      'valueType': 'long'}},
            {'metadata': {},
             'name': 'array_example',
             'nullable': True,
             'type': {'containsNull': True,
                      'elementType': 'string',
                      'type': 'array'}}],
 'type': 'struct'}


In [61]:
other_document_schema = T.StructType.fromJson(
    json.loads(document_with_schema.schema.json())
)

other_document_schema == document_with_schema.schema

True

## **Explode and Collect**

In [67]:
document.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

In [65]:
episodes = document.select(
    'id', F.explode('_embedded.episodes').alias('episodes')
)

episodes.show(5, truncate=70)

+---+----------------------------------------------------------------------+
| id|                                                              episodes|
+---+----------------------------------------------------------------------+
|143|{{{http://api.tvmaze.com/episodes/10897}}, 2014-04-06, 2014-04-07T0...|
|143|{{{http://api.tvmaze.com/episodes/10898}}, 2014-04-13, 2014-04-14T0...|
|143|{{{http://api.tvmaze.com/episodes/10899}}, 2014-04-20, 2014-04-21T0...|
|143|{{{http://api.tvmaze.com/episodes/10900}}, 2014-04-27, 2014-04-28T0...|
|143|{{{http://api.tvmaze.com/episodes/10901}}, 2014-05-04, 2014-05-05T0...|
+---+----------------------------------------------------------------------+
only showing top 5 rows


In [66]:
episodes.count()

53

In [70]:
episode_name_id = document.select(
    F.map_from_arrays(
        F.col("_embedded.episodes.id"), F.col("_embedded.episodes.name")
    ).alias('name_id')
)

episode_name_id.show(1, False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [71]:
episode_name_id = episode_name_id.select(
    F.posexplode('name_id').alias('position', 'id', 'name')
)

episode_name_id.show(5)

+--------+-----+--------------------+
|position|   id|                name|
+--------+-----+--------------------+
|       0|10897|Minimum Viable Pr...|
|       1|10898|       The Cap Table|
|       2|10899|Articles of Incor...|
|       3|10900|    Fiduciary Duties|
|       4|10901|      Signaling Risk|
+--------+-----+--------------------+
only showing top 5 rows


> Both **`explode()`** and **`posexplode()`** will skip any null values in the array or the map. If you want to have null as records, you can use **`explode_outer()`** or **`posexplode_outer()`** the same way.

In [74]:
episodes.show(5)

+---+--------------------+
| id|            episodes|
+---+--------------------+
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
+---+--------------------+
only showing top 5 rows


In [75]:
collected_episodes = episodes.groupby('id').agg(
    F.collect_list('episodes').alias('episodes')
)

collected_episodes.show()

+---+--------------------+
| id|            episodes|
+---+--------------------+
|143|[{{{http://api.tv...|
+---+--------------------+



In [76]:
collected_episodes.count()

1

In [77]:
collected_episodes.printSchema()

root
 |-- id: long (nullable = true)
 |-- episodes: array (nullable = false)
 |    |-- element: struct (containsNull = false)
 |    |    |-- _links: struct (nullable = true)
 |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |-- href: string (nullable = true)
 |    |    |-- airdate: string (nullable = true)
 |    |    |-- airstamp: string (nullable = true)
 |    |    |-- airtime: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- image: struct (nullable = true)
 |    |    |    |-- medium: string (nullable = true)
 |    |    |    |-- original: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- number: long (nullable = true)
 |    |    |-- runtime: long (nullable = true)
 |    |    |-- season: long (nullable = true)
 |    |    |-- summary: string (nullable = true)
 |    |    |-- url: string (nullable = true)



In [78]:
struct_ex = document.select(
    F.struct(
        F.col("status"), F.col("weight"), F.lit(True).alias("has_watched")
    ).alias('info')
)

In [79]:
struct_ex.show(1, False)

+-----------------+
|info             |
+-----------------+
|{Ended, 96, true}|
+-----------------+



In [80]:
struct_ex.printSchema()

root
 |-- info: struct (nullable = false)
 |    |-- status: string (nullable = true)
 |    |-- weight: long (nullable = true)
 |    |-- has_watched: boolean (nullable = false)

